# PixelRAG — Self-Contained Notebook

Visual document retrieval and Q&A — no text parsing, everything runs here.

**Pipeline:**  PDF → page images → embed (Qwen3-VL on Colab GPU) → FAISS + MMR → rerank (gpt-4o) → crop → answer

**Requirements before running:**
1. Upload these 4 files to **Google Drive** at `MyDrive/PixleRAG_notebbok/`:
   - `requirements.txt`
   - `config.yaml`
   - `.env` (needs only `OPENAI_API_KEY`)
   - this notebook
2. Run cells **top to bottom** — Drive mount must happen before config is loaded

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# requirements.txt must be in the same Google Drive folder as this notebook
%pip install -q -r requirements.txt

In [ ]:
# ── Cell 1b: Mount Google Drive and set working directory ─────────────────────
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

# ── Set this to your project folder inside Google Drive ───────────────────────
PROJECT_DIR = "/content/drive/MyDrive/PixleRAG_notebbok"   # ← change if needed

os.chdir(PROJECT_DIR)
print(f"✅ Working directory: {os.getcwd()}")
print(f"   Files found: {[f.name for f in Path('.').iterdir() if not f.name.startswith('.')]}")

In [ ]:
# ── Cell 2: Load config.yaml and .env ─────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
import yaml

# Secrets from .env
load_dotenv()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
JINA_API_KEY   = os.environ.get("JINA_API_KEY", "")

assert OPENAI_API_KEY, "❌ OPENAI_API_KEY not set in .env"

# Settings from config.yaml
_cfg = yaml.safe_load(Path("config.yaml").read_text())

RERANKER     = str(_cfg["reranker"]).strip()
TOP_K        = int(_cfg["top_k"])
MMR_LAMBDA   = float(_cfg["mmr_lambda"])
RERANK_TOP_N = int(_cfg["rerank_top_n"])
PDF_DPI      = int(_cfg["pdf_dpi"])
MAX_TILE_PX  = int(_cfg["max_tile_px"])
ANSWER_MODEL = str(_cfg["answer_model"])
CROP_MIN_PX  = int(_cfg["crop_min_px"])

# Index storage — notebook uses its own folder to avoid colliding with the app
DATA_DIR  = Path("nb_data")
TILES_DIR = DATA_DIR / "tiles"
DATA_DIR.mkdir(exist_ok=True)
TILES_DIR.mkdir(exist_ok=True)

print("✅ Config loaded")
print(f"   Reranker     : {RERANKER}")
print(f"   Answer model : {ANSWER_MODEL}")
print(f"   PDF DPI      : {PDF_DPI}  |  MMR lambda: {MMR_LAMBDA}  |  TOP_K: {TOP_K}")

In [ ]:
# ── Cell 3: Load embedding model directly on Colab GPU ────────────────────────
import torch
import numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModel

EMBED_MODEL_ID = "tomaarsen/Qwen3-VL-Embedding-2B-vdr"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {EMBED_MODEL_ID} on {DEVICE} ...")
_embed_processor = AutoProcessor.from_pretrained(EMBED_MODEL_ID, trust_remote_code=True)
_embed_model     = AutoModel.from_pretrained(
    EMBED_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE).eval()

def _last_token_pool(last_hidden_state, attention_mask):
    """Use the last real token (EOS) as the embedding — standard for VDR models."""
    last_idx = attention_mask.sum(dim=1) - 1
    return last_hidden_state[torch.arange(last_hidden_state.size(0), device=last_hidden_state.device), last_idx]

def embed_image(img: Image.Image) -> list:
    """Embed a PIL image → normalised float32 vector."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "Represent this document page for retrieval."},
        ],
    }]
    text = _embed_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _embed_processor(text=[text], images=[img], return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        out = _embed_model(**inputs)
    vec = _last_token_pool(out.last_hidden_state, inputs["attention_mask"])
    vec = torch.nn.functional.normalize(vec, dim=-1)
    return vec[0].cpu().float().tolist()

def embed_text(text: str) -> list:
    """Embed a text query → normalised float32 vector."""
    messages = [{"role": "user", "content": [{"type": "text", "text": text}]}]
    prompt = _embed_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _embed_processor(text=[prompt], return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        out = _embed_model(**inputs)
    vec = _last_token_pool(out.last_hidden_state, inputs["attention_mask"])
    vec = torch.nn.functional.normalize(vec, dim=-1)
    return vec[0].cpu().float().tolist()

print(f"✅ Embedding model ready  ({DEVICE})")

In [ ]:
# ── Cell 4: Ingest ─────────────────────────────────────────────────────────────
import json
import faiss
import numpy as np
import fitz  # pymupdf

INDEX_PATH = DATA_DIR / "index.faiss"
META_PATH  = DATA_DIR / "metadata.json"

def _load_meta():
    return json.loads(META_PATH.read_text()) if META_PATH.exists() else []

def _save_meta(meta):
    META_PATH.write_text(json.dumps(meta, indent=2))

def _load_index(dim):
    return faiss.read_index(str(INDEX_PATH)) if INDEX_PATH.exists() else faiss.IndexFlatIP(dim)

def _save_index(index):
    faiss.write_index(index, str(INDEX_PATH))

def _resize(img):
    w, h = img.size
    if max(w, h) <= MAX_TILE_PX: return img
    s = MAX_TILE_PX / max(w, h)
    return img.resize((int(w*s), int(h*s)), Image.LANCZOS)

def ingest_file(file_path, progress_cb=print):
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()
    pages = []
    progress_cb(f"Rendering {file_path.name} at {PDF_DPI} DPI...")
    if suffix == ".pdf":
        doc = fitz.open(str(file_path))
        mat = fitz.Matrix(PDF_DPI/72, PDF_DPI/72)
        for i in range(len(doc)):
            pix = doc[i].get_pixmap(matrix=mat, alpha=False)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            pages.append((i+1, img))
        doc.close()
    elif suffix in {".png",".jpg",".jpeg",".webp",".bmp",".tiff"}:
        pages.append((1, Image.open(file_path).convert("RGB")))
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

    meta = _load_meta()
    new_vecs, new_meta = [], []
    dim = None

    for page_num, img in pages:
        img = _resize(img)
        tile_path = TILES_DIR / f"{file_path.stem}_p{page_num:04d}.png"
        img.save(tile_path, format="PNG")
        progress_cb(f"  Embedding page {page_num}/{len(pages)}...")
        vec = embed_image(img)
        new_vecs.append(np.array(vec, dtype=np.float32))
        new_meta.append({"source": file_path.name, "page": page_num, "tile_path": str(tile_path)})
        if dim is None: dim = len(vec)

    index = _load_index(dim)
    start_id = index.ntotal
    index.add(np.stack(new_vecs))
    for i, m in enumerate(new_meta):
        m["vector_id"] = start_id + i
        meta.append(m)
    _save_index(index)
    _save_meta(meta)
    progress_cb(f"  ✅ Indexed {len(new_vecs)} page(s) from {file_path.name}")
    return len(new_vecs)

def clear_index():
    for p in [INDEX_PATH, META_PATH]:
        if p.exists(): p.unlink()
    for f in TILES_DIR.glob("*.png"): f.unlink()
    print("✅ Index cleared")

def list_indexed_files():
    seen = {}
    for m in _load_meta():
        seen.setdefault(m["source"], 0)
        seen[m["source"]] += 1
    return [{"source": k, "pages": v} for k, v in seen.items()]

print("✅ Ingest functions ready")

In [ ]:
# ── Cell 5: Search (FAISS + MMR) ───────────────────────────────────────────────

class SearchResult:
    def __init__(self, vector_id, score, source, page, tile_path):
        self.vector_id = vector_id
        self.score     = score
        self.source    = source
        self.page      = page
        self.tile_path = tile_path
        self._image    = None

    @property
    def image(self):
        if self._image is None and self.tile_path and Path(self.tile_path).exists():
            self._image = Image.open(self.tile_path).convert("RGB")
        return self._image

    def __repr__(self):
        return f"SearchResult(page={self.page}, score={self.score:.4f})"


def _mmr(query_vec, candidate_vecs, candidate_scores, top_k, lam=MMR_LAMBDA):
    n = len(candidate_scores)
    top_k = min(top_k, n)
    scores = np.array(candidate_scores, dtype=np.float32)
    s_min, s_max = scores.min(), scores.max()
    rel = (scores - s_min) / (s_max - s_min + 1e-9)
    selected, remaining = [], list(range(n))
    while len(selected) < top_k and remaining:
        if not selected:
            best = max(remaining, key=lambda i: rel[i])
        else:
            sel_vecs = candidate_vecs[selected]
            sims = candidate_vecs[remaining] @ sel_vecs.T
            max_sim = sims.max(axis=1)
            mmr_s = lam * rel[remaining] - (1 - lam) * max_sim
            best = remaining[int(np.argmax(mmr_s))]
        selected.append(best)
        remaining.remove(best)
    return selected


def search(query, top_k=TOP_K):
    if not INDEX_PATH.exists():
        raise FileNotFoundError("No index found — run ingest first.")
    vec = embed_text(query)
    q   = np.array(vec, dtype=np.float32).reshape(1, -1)
    index = faiss.read_index(str(INDEX_PATH))
    meta  = {m["vector_id"]: m for m in _load_meta()}
    n_candidates = min(top_k * 10, index.ntotal)
    scores, ids = index.search(q, n_candidates)
    candidates = []
    for s, vid in zip(scores[0], ids[0]):
        if vid == -1: continue
        m = meta.get(int(vid))
        if m: candidates.append({"score": float(s), "meta": m, "vid": int(vid)})
    if not candidates: return []
    vecs   = np.stack([index.reconstruct(c["vid"]) for c in candidates]).astype(np.float32)
    mmr_ix = _mmr(q, vecs, [c["score"] for c in candidates], top_k)
    return [
        SearchResult(candidates[i]["vid"], candidates[i]["score"],
                     candidates[i]["meta"]["source"], candidates[i]["meta"]["page"],
                     candidates[i]["meta"]["tile_path"])
        for i in mmr_ix
    ]

print("✅ Search functions ready")

In [ ]:
# ── Cell 6: Answer synthesis (rerank + crop + gpt-4o) ─────────────────────────
import base64
import io
import re
import requests as req_lib
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)

def _img_to_data_url(img, detail="high"):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}", "detail": detail}}

def _has_figure_ref(query):
    return bool(re.search(r"\b(figure|fig|table|chart|graph|diagram)\s*[\d]+", query, re.I))

def _rerank_gpt4o(query, results):
    content = [{"type": "text", "text": (
        f"I will show you {len(results)} document page images numbered 1 to {len(results)}.\n"
        f"Question: {query}\n\n"
        "Return ONLY a JSON array of image numbers that are relevant, most to least relevant. "
        "Example: [3, 1, 5]. If none: []"
    )}]
    for i, r in enumerate(results):
        img = r.image
        if img is None: continue
        content.append({"type": "text", "text": f"Image {i+1} — {r.source}, page {r.page}:"})
        content.append(_img_to_data_url(img, detail="low"))
    resp = openai_client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=[{"role": "user", "content": content}],
        max_tokens=128,
    )
    raw = resp.choices[0].message.content.strip()
    start, end = raw.find("["), raw.rfind("]")
    ranked = json.loads(raw[start:end+1]) if start != -1 else []
    ranked = [int(i)-1 for i in ranked if 1 <= int(i) <= len(results)]
    seen = set(ranked)
    return [results[i] for i in ranked] + [r for i,r in enumerate(results) if i not in seen]

def _rerank_jina(query, results):
    docs, idx = [], []
    for i, r in enumerate(results):
        img = r.image
        if img is None: continue
        buf = io.BytesIO(); img.save(buf, format="PNG")
        docs.append({"image": f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"})
        idx.append(i)
    resp = req_lib.post(
        "https://api.jina.ai/v1/rerank",
        json={"model": "jina-reranker-v2-base-multimodal", "query": query,
              "documents": docs, "top_n": len(docs)},
        headers={"Authorization": f"Bearer {JINA_API_KEY}", "Content-Type": "application/json"},
        timeout=30,
    )
    resp.raise_for_status()
    scored = sorted(resp.json()["results"], key=lambda x: x["relevance_score"], reverse=True)
    ranked = [idx[item["index"]] for item in scored]
    seen = set(ranked)
    return [results[i] for i in ranked] + [r for i,r in enumerate(results) if i not in seen]

def _rerank(query, results):
    if len(results) <= 1: return results
    if RERANKER == "jina-reranker-v2-base-multimodal":
        if not JINA_API_KEY: raise ValueError("JINA_API_KEY not set")
        try: return _rerank_jina(query, results)
        except Exception as e:
            print(f"[rerank] Jina failed ({e}), falling back to gpt-4o")
    return _rerank_gpt4o(query, results)

def _locate_and_crop(query, img):
    prompt = (
        f"Question: {query}\n\nIdentify the specific figure on this page relevant to the question.\n"
        "Return ONLY JSON: {\"top\": n, \"left\": n, \"bottom\": n, \"right\": n} (0-100% of page).\n"
        "If not found: {}"
    )
    resp = openai_client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=[{"role": "user", "content": [_img_to_data_url(img, "high"), {"type": "text", "text": prompt}]}],
        max_tokens=64,
    )
    raw = resp.choices[0].message.content.strip()
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1: return None
    box = json.loads(raw[start:end+1])
    if not box or not all(k in box for k in ("top","left","bottom","right")): return None
    if (box["bottom"]-box["top"]) < 5 or (box["right"]-box["left"]) < 5: return None
    w, h = img.size
    px, py = int(0.02*w), int(0.02*h)
    l = max(0, int(box["left"]/100*w)-px);  t = max(0, int(box["top"]/100*h)-py)
    r = min(w, int(box["right"]/100*w)+px); b = min(h, int(box["bottom"]/100*h)+py)
    crop = img.crop((l, t, r, b))
    cw, ch = crop.size
    if cw < CROP_MIN_PX:
        s = CROP_MIN_PX / cw
        crop = crop.resize((int(cw*s), int(ch*s)), Image.LANCZOS)
    return crop

def ask(query, top_k=TOP_K):
    results  = search(query, top_k=top_k)
    reranked = _rerank(query, results)
    top      = reranked[:RERANK_TOP_N]

    crop = None
    if _has_figure_ref(query) and top:
        img = top[0].image
        if img: crop = _locate_and_crop(query, img)

    content = []
    if crop:
        content += [
            {"type": "text", "text": f"[Zoomed crop — {top[0].source}, page {top[0].page}]"},
            _img_to_data_url(crop, "high"),
            {"type": "text", "text": f"[Full page — {top[0].source}, page {top[0].page}]"},
            _img_to_data_url(top[0].image, "high"),
        ]
        for i, r in enumerate(top[1:]):
            img = r.image
            if img:
                content += [{"type": "text", "text": f"[Image {i+2}: {r.source}, page {r.page}]"}, _img_to_data_url(img, "high")]
    else:
        for i, r in enumerate(top):
            img = r.image
            if img:
                content += [{"type": "text", "text": f"[Image {i+1}: {r.source}, page {r.page}]"}, _img_to_data_url(img, "high")]

    content.append({"type": "text", "text": f"\nQuestion: {query}"})

    system = (
        "You are a precise document analyst. "
        + ("The first image is a zoomed crop of the relevant figure — use it for fine details. " if crop else "")
        + "Answer using ONLY what is visible in the images. Be specific. "
        "Read alphanumeric codes character by character, preserving all hyphens. "
        "Cite which image(s) you used. If the answer is not in the images, say so."
    )
    response = openai_client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": content}],
        max_tokens=1024,
    )
    return response.choices[0].message.content, reranked

print("✅ Answer synthesis ready")

---
## Usage
Run the cells above once to load everything, then use the cells below.

In [ ]:
# ── Ingest a document ──────────────────────────────────────────────────────────
FILE = "ocr_evaluation_test.pdf"   # ← change to your file path

ingest_file(FILE)

In [ ]:
# ── List indexed files ─────────────────────────────────────────────────────────
for f in list_indexed_files():
    print(f"  • {f['source']}  ({f['pages']} pages)")

In [ ]:
# ── Search only (no answer) ────────────────────────────────────────────────────
QUERY   = "Which product line had the highest Q4 revenue?"   # ← change me
results = search(QUERY)

for i, r in enumerate(results):
    print(f"[{i+1}] page {r.page}  score={r.score:.4f}  {r.source}")

In [ ]:
# ── Display retrieved pages ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5*n, 7))
if n == 1: axes = [axes]
for ax, r in zip(axes, results):
    img = r.image
    if img: ax.imshow(img)
    ax.set_title(f"page {r.page}  {r.score:.3f}", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── Ask a question (full pipeline) ────────────────────────────────────────────
QUESTION = "Which product line had the highest Q4 revenue?"   # ← change me

answer, reranked = ask(QUESTION)

print("Question:", QUESTION)
print()
print("Answer:")
print(answer)

In [ ]:
# ── Batch Q&A → save to JSON ───────────────────────────────────────────────────
from datetime import datetime

QUESTIONS = [
    "What is the recommended minimum scan DPI for printed documents?",
    "What is the Character Error Rate of DocLang v2.0?",
    "In Figure 1, which product line had the highest Q4 revenue?",
    "In Figure 3, what percentage of the test set is printed text?",
    # ← add more
]

log = []
for i, q in enumerate(QUESTIONS):
    print(f"[{i+1}/{len(QUESTIONS)}] {q}")
    ans, res = ask(q)
    log.append({
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "question": q, "answer": ans,
        "retrieved": [{"source": r.source, "page": r.page, "score": round(float(r.score),4)} for r in res],
    })
    print(f"   → {ans[:120]}\n")

out = DATA_DIR / "qa_log.json"
out.write_text(json.dumps(log, indent=2))
print(f"\n✅ Saved {len(log)} Q&A pairs → {out}")

In [ ]:
# ── Clear index (destructive) ──────────────────────────────────────────────────
# Uncomment and run only when you want a fresh start
# clear_index()